# Day 5：经典评价指标 — BLEU / ROUGE / PPL 数学推导与手写实现

> **目标**：理解三大经典评价指标的数学定义、适用场景与局限性，并从零实现。

---

## 为什么需要评价指标？

训练一个语言模型后，我们需要量化评估其好坏。不同任务使用不同指标：

| 指标 | 全称 | 适用场景 | 核心思想 |
|------|------|---------|----------|
| **BLEU** | Bilingual Evaluation Understudy | 翻译、文本生成 | 生成结果与参考答案的 n-gram 重合度 |
| **ROUGE** | Recall-Oriented Understudy for Gisting Evaluation | 摘要 | 参考答案中的 n-gram 被生成结果覆盖的比例 |
| **PPL** | Perplexity | 语言模型质量 | 模型对测试文本的「困惑度」 |

## 一、BLEU（Bilingual Evaluation Understudy）

### 1.1 数学定义

BLEU 衡量的是：**生成文本中的 n-gram 有多少出现在参考文本中？**（Precision 导向）

#### Step 1：Modified n-gram Precision

普通 precision 的问题：如果生成 `"the the the the"` 而参考是 `"the cat sat on the mat"`，unigram precision = 4/4 = 100%，但这显然不合理。

**Modified Precision**：每个 n-gram 的匹配次数不超过它在参考中出现的最大次数。

$$
p_n = \frac{\sum_{\text{n-gram} \in C} \min\big(\text{Count}(\text{n-gram}, C),\ \max_{r \in R} \text{Count}(\text{n-gram}, r)\big)}{\sum_{\text{n-gram} \in C} \text{Count}(\text{n-gram}, C)}
$$

其中 $C$ 是 candidate（生成文本），$R$ 是 reference（参考文本集合）。

#### Step 2：Brevity Penalty (BP)

BLEU 是 precision-based，短文本天然 precision 高。为了惩罚过短的输出：

$$
\text{BP} = \begin{cases} 1 & \text{if } c > r \\ e^{1 - r/c} & \text{if } c \leq r \end{cases}
$$

其中 $c$ = candidate 长度，$r$ = 最接近 $c$ 的 reference 长度。

#### Step 3：最终 BLEU 分数

$$
\text{BLEU-N} = \text{BP} \cdot \exp\!\left(\sum_{n=1}^{N} w_n \log p_n\right)
$$

标准 BLEU-4：$N=4$，$w_n = 1/4$（等权几何平均）。

In [ ]:
from collections import Counter
import math


def get_ngrams(tokens: list[str], n: int) -> Counter:
    """提取 n-gram 并统计频次。"""
    return Counter(tuple(tokens[i:i+n]) for i in range(len(tokens) - n + 1))


def compute_bleu(
    candidate: str,
    references: list[str],
    max_n: int = 4,
    weights: list[float] | None = None,
) -> dict:
    """
    从零实现 BLEU 分数。
    
    Args:
        candidate: 模型生成的文本
        references: 一个或多个参考文本
        max_n: 最大 n-gram 阶数
        weights: 各阶权重，默认等权
    """
    if weights is None:
        weights = [1.0 / max_n] * max_n

    cand_tokens = candidate.lower().split()
    ref_tokens_list = [ref.lower().split() for ref in references]

    # Step 1: 计算各阶 modified precision
    precisions = []
    for n in range(1, max_n + 1):
        cand_ngrams = get_ngrams(cand_tokens, n)
        
        # 对每个 n-gram，取所有 reference 中出现次数的最大值作为 clip 上限
        max_ref_counts = Counter()
        for ref_tokens in ref_tokens_list:
            ref_ngrams = get_ngrams(ref_tokens, n)
            for ngram, count in ref_ngrams.items():
                max_ref_counts[ngram] = max(max_ref_counts[ngram], count)
        
        # Clipped count
        clipped_count = 0
        total_count = 0
        for ngram, count in cand_ngrams.items():
            clipped_count += min(count, max_ref_counts.get(ngram, 0))
            total_count += count
        
        precision = clipped_count / total_count if total_count > 0 else 0
        precisions.append(precision)

    # Step 2: Brevity Penalty
    c = len(cand_tokens)
    ref_lens = [len(ref) for ref in ref_tokens_list]
    r = min(ref_lens, key=lambda x: abs(x - c))  # 最接近 candidate 长度的 reference 长度
    
    if c > r:
        bp = 1.0
    elif c == 0:
        bp = 0.0
    else:
        bp = math.exp(1 - r / c)

    # Step 3: 加权几何平均
    log_avg = 0.0
    for w, p in zip(weights, precisions):
        if p == 0:
            log_avg = float('-inf')
            break
        log_avg += w * math.log(p)

    bleu = bp * math.exp(log_avg) if log_avg != float('-inf') else 0.0

    return {
        'bleu': bleu,
        'precisions': precisions,
        'bp': bp,
        'candidate_len': c,
        'reference_len': r,
    }

In [ ]:
# 测试 BLEU 实现
candidate = "the cat sat on the mat"
references = ["the cat is on the mat", "there is a cat on the mat"]

result = compute_bleu(candidate, references)
print(f"候选: '{candidate}'")
print(f"参考: {references}")
print(f"\nBLEU-4: {result['bleu']:.4f}")
print(f"各阶 Precision: {[f'{p:.4f}' for p in result['precisions']]}")
print(f"Brevity Penalty: {result['bp']:.4f}")
print(f"候选长度: {result['candidate_len']}, 参考长度: {result['reference_len']}")

print("\n" + "="*60)

# 对比：差的翻译
bad_candidate = "the the the the the the"
result_bad = compute_bleu(bad_candidate, references)
print(f"\n差的候选: '{bad_candidate}'")
print(f"BLEU-4: {result_bad['bleu']:.4f}")
print(f"各阶 Precision: {[f'{p:.4f}' for p in result_bad['precisions']]}")
print("→ Modified Precision 成功惩罚了重复输出!")

### 1.2 BLEU 的局限性

1. **不考虑语义**：同义词替换会降低 BLEU（`"cat"` vs `"feline"`）
2. **不考虑语法**：打乱词序也可能有高 BLEU
3. **Precision 偏向**：即使加了 BP，仍然偏向短输出
4. **对话/开放式生成不适用**：有意义的回复不一定与参考重叠

**适用场景**：机器翻译等有明确参考答案的任务。

## 二、ROUGE（Recall-Oriented Understudy for Gisting Evaluation）

### 2.1 ROUGE 与 BLEU 的关键区别

| | BLEU | ROUGE |
|--|------|-------|
| 导向 | **Precision** | **Recall** |
| 问题 | 生成的 n-gram 多少在参考中？ | 参考的 n-gram 多少被生成覆盖？ |
| 适用 | 翻译 | 摘要 |

为什么摘要关注 Recall？因为我们关心的是：**参考摘要中的关键信息有没有被生成出来。**

### 2.2 ROUGE 家族

#### ROUGE-N（n-gram Recall）

$$
\text{ROUGE-N} = \frac{\sum_{\text{n-gram} \in \text{Ref}} \text{Count}_{\text{match}}(\text{n-gram})}{\sum_{\text{n-gram} \in \text{Ref}} \text{Count}(\text{n-gram})}
$$

常用 ROUGE-1（unigram）和 ROUGE-2（bigram）。

#### ROUGE-L（最长公共子序列）

基于 LCS（Longest Common Subsequence）而非 n-gram：

$$
R_{\text{lcs}} = \frac{\text{LCS}(X, Y)}{m}, \quad
P_{\text{lcs}} = \frac{\text{LCS}(X, Y)}{n}
$$
$$
F_{\text{lcs}} = \frac{(1 + \beta^2) \cdot R_{\text{lcs}} \cdot P_{\text{lcs}}}{R_{\text{lcs}} + \beta^2 \cdot P_{\text{lcs}}}
$$

其中 $m$ = 参考长度，$n$ = 生成长度，$\beta$ 控制 R vs P 的权重（通常 $\beta$ 很大 → Recall 为主）。

**LCS 的优势**：不要求 n-gram 连续，能捕捉更灵活的语序变化。

In [ ]:
def compute_rouge_n(
    candidate: str,
    reference: str,
    n: int = 1,
) -> dict:
    """计算 ROUGE-N (Precision, Recall, F1)。"""
    cand_tokens = candidate.lower().split()
    ref_tokens = reference.lower().split()

    cand_ngrams = get_ngrams(cand_tokens, n)  # 复用 BLEU 部分的 get_ngrams
    ref_ngrams = get_ngrams(ref_tokens, n)

    # 计算重叠的 n-gram 数
    overlap = 0
    for ngram, count in ref_ngrams.items():
        overlap += min(count, cand_ngrams.get(ngram, 0))

    total_ref = sum(ref_ngrams.values())
    total_cand = sum(cand_ngrams.values())

    recall = overlap / total_ref if total_ref > 0 else 0
    precision = overlap / total_cand if total_cand > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

    return {'precision': precision, 'recall': recall, 'f1': f1}


def lcs_length(x: list, y: list) -> int:
    """动态规划计算最长公共子序列长度。"""
    m, n = len(x), len(y)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if x[i-1] == y[j-1]:
                dp[i][j] = dp[i-1][j-1] + 1
            else:
                dp[i][j] = max(dp[i-1][j], dp[i][j-1])
    return dp[m][n]


def compute_rouge_l(
    candidate: str,
    reference: str,
    beta: float = 1.2,
) -> dict:
    """计算 ROUGE-L（基于 LCS）。"""
    cand_tokens = candidate.lower().split()
    ref_tokens = reference.lower().split()

    lcs = lcs_length(ref_tokens, cand_tokens)

    recall = lcs / len(ref_tokens) if len(ref_tokens) > 0 else 0
    precision = lcs / len(cand_tokens) if len(cand_tokens) > 0 else 0

    if recall == 0 and precision == 0:
        f1 = 0
    else:
        f1 = ((1 + beta**2) * precision * recall) / (recall + beta**2 * precision)

    return {'precision': precision, 'recall': recall, 'f1': f1, 'lcs': lcs}

In [ ]:
# 测试 ROUGE
candidate = "the cat was found under the bed"
reference = "the cat was under the bed"

print(f"候选: '{candidate}'")
print(f"参考: '{reference}'")
print()

for n in [1, 2]:
    result = compute_rouge_n(candidate, reference, n=n)
    print(f"ROUGE-{n}: P={result['precision']:.4f}, R={result['recall']:.4f}, F1={result['f1']:.4f}")

result_l = compute_rouge_l(candidate, reference)
print(f"ROUGE-L: P={result_l['precision']:.4f}, R={result_l['recall']:.4f}, F1={result_l['f1']:.4f} (LCS={result_l['lcs']})") 

print("\n" + "="*60)

# 关键直觉：ROUGE 关注 Recall
cand_short = "the cat"
cand_long = "the cat was under the bed and the dog was on the rug and all the animals slept"

print(f"\n短候选 ROUGE-1: {compute_rouge_n(cand_short, reference, 1)}")
print(f"长候选 ROUGE-1: {compute_rouge_n(cand_long, reference, 1)}")
print("→ 长候选 Recall 更高（覆盖了参考中更多内容），但 Precision 更低")

## 三、Perplexity（困惑度）

### 3.1 数学定义

Perplexity 衡量的是：**语言模型对测试数据有多「困惑」。** 困惑度越低，模型越好。

给定测试文本 $W = w_1, w_2, \ldots, w_N$，语言模型的困惑度为：

$$
\text{PPL}(W) = P(w_1, w_2, \ldots, w_N)^{-1/N}
$$

展开为自回归形式：

$$
\text{PPL}(W) = \exp\!\left(-\frac{1}{N} \sum_{i=1}^{N} \log P(w_i \mid w_{<i})\right)
$$

即 **cross-entropy loss 的指数**：

$$
\text{PPL} = e^{H(W)} = e^{\text{CE Loss}}
$$

### 3.2 直觉理解

- PPL = 1：模型对每个 token 都 100% 确定 → 完美预测
- PPL = V（词表大小）：模型对每个 token 等概率猜测 → 随机
- **PPL 可以理解为：模型在每个位置平均需要在多少个候选中做选择**

| 模型 | 大致 PPL | 说明 |
|------|---------|------|
| 随机猜测 (V=50K) | 50,000 | 最差 |
| 早期 LSTM LM | 50-80 | |
| GPT-2 (1.5B) | ~20 | |
| GPT-3 (175B) | ~10 | |
| 当前最优 | 3-8 | 取决于测试集 |

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np


def compute_perplexity_from_logits(
    logits: torch.Tensor,
    target_ids: torch.Tensor,
) -> float:
    """
    从模型 logits 计算 perplexity。
    
    Args:
        logits: 模型输出 (seq_len, vocab_size)
        target_ids: 目标 token IDs (seq_len,)
    """
    # Cross-entropy loss (平均)
    ce_loss = F.cross_entropy(logits, target_ids, reduction='mean')
    ppl = torch.exp(ce_loss).item()
    return ppl


def compute_perplexity_from_probs(
    token_probs: list[float],
) -> float:
    """
    从每个 token 的预测概率计算 perplexity。
    PPL = exp(-1/N * sum(log(p_i)))
    """
    N = len(token_probs)
    log_prob_sum = sum(math.log(p) for p in token_probs)
    ppl = math.exp(-log_prob_sum / N)
    return ppl

In [ ]:
# 示例 1：从概率列表计算 PPL
# 假设模型对每个 token 的预测概率
high_confidence = [0.9, 0.85, 0.92, 0.88, 0.95]  # 模型很确定
low_confidence = [0.1, 0.15, 0.08, 0.12, 0.05]    # 模型很困惑
random_guess = [1/50000] * 5                        # 完全随机猜测

print("从概率计算 PPL：")
print(f"  高置信度: PPL = {compute_perplexity_from_probs(high_confidence):.2f}")
print(f"  低置信度: PPL = {compute_perplexity_from_probs(low_confidence):.2f}")
print(f"  随机猜测: PPL = {compute_perplexity_from_probs(random_guess):.2f}")

print("\n" + "="*60)

# 示例 2：模拟模型 logits 计算 PPL
torch.manual_seed(42)
vocab_size = 50000
seq_len = 20

# 好模型：target token 的 logit 很高
target_ids = torch.randint(0, vocab_size, (seq_len,))
logits_good = torch.randn(seq_len, vocab_size) * 0.1
for i in range(seq_len):
    logits_good[i, target_ids[i]] += 5.0  # 大幅提高目标 token 的 logit

# 差模型：随机 logits
logits_bad = torch.randn(seq_len, vocab_size) * 0.1

print("从 logits 计算 PPL：")
print(f"  好模型: PPL = {compute_perplexity_from_logits(logits_good, target_ids):.2f}")
print(f"  差模型: PPL = {compute_perplexity_from_logits(logits_bad, target_ids):.2f}")

### 3.3 PPL 的注意事项

1. **PPL 只能在相同词表的模型间比较**：不同 tokenizer 的 PPL 不可直接对比（token 粒度不同）
2. **PPL 只反映语言建模能力**：PPL 低不等于下游任务好（一个背诵维基百科的模型 PPL 很低，但不一定能做推理）
3. **PPL 与 CE Loss 的关系**：PPL = exp(CE Loss)，所以训练时看 loss 等价于看 PPL
4. **每个 token 的贡献不同**：功能词（the, is）通常容易预测，PPL 低；实义词（Paris, quantum）更难预测

## 四、三大指标对比总结

| 维度 | BLEU | ROUGE | PPL |
|------|------|-------|-----|
| 导向 | Precision | Recall | 语言模型内在质量 |
| 需要参考答案 | ✅ | ✅ | ❌（只需测试文本） |
| 适用任务 | 翻译 | 摘要 | 语言模型评估 |
| 值域 | [0, 1] | [0, 1] | [1, V] |
| 越大/小越好 | 越大越好 | 越大越好 | **越小越好** |
| 局限性 | 不考虑语义 | 不考虑语义 | 不反映任务能力 |

### LLM 时代的评估趋势

这些经典指标在 LLM 时代的局限愈发明显：
- 开放式生成没有标准答案 → BLEU/ROUGE 不适用
- PPL 不能反映对话质量和指令遵循能力

因此，现代评估转向了 **基准测试**（MMLU, HumanEval）和 **人类/模型评估**（MT-Bench, Arena）—— Day 6 详解。

## 五、练习题

### 练习 1：手动计算 BLEU
给定 candidate = `"the cat sat"` 和 reference = `"the cat sat on the mat"`，手算 BLEU-2 (只考虑 unigram 和 bigram)，包括 BP。

### 练习 2：BLEU vs ROUGE 对比
构造一个例子，使得 BLEU 很高但 ROUGE-1 Recall 很低（提示：生成很短但词都对）。再构造一个反例。

### 练习 3：PPL 与 Loss 换算
如果一个模型的 cross-entropy loss 是 2.5，其 PPL 是多少？如果想把 PPL 从 20 降到 10，loss 需要降低多少？

### 练习 4（选做）：用 HuggingFace 模型计算 PPL
加载一个预训练 GPT-2 模型，计算其在一段中文/英文文本上的 PPL。